---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
%pip install google-genai

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [8]:
import pandas as pd
import random

corpus = pd.read_json(r"C:\Users\ASUS\Desktop\ADC 2\INGINERIE AI\echochamber-project-team-2\data\cleaned\corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[RecorderRomania] Probabil se mai mergea vreo 20 de ani prin praf si noroi daca avea sa obtina toa
[@CălinGeorgescu-CanalulOficial] ARE DREPTATE DOMNUL CĂLIN GEORGESCU CÂND ȚARA ARE NEVOIE DE AJUTOR CEI CU BANI A
[TuDecizi-s3g] Aloooo,AUR ati gesit sistematic! De ce nu cere re numararea voturilor?🤷🏻‍♀️


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [9]:
# modifica dupa preferinte

AXA_1 = "people_vs_elite"
AXA_2 = "fear"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [10]:
AXA_1_DEFINITION = """
people_vs_elite măsoară dacă textul construiește o opoziție între „popor/oameni”
și „elite/sistem/clasa conducătoare”.
0 = absent
1 = slab
2 = moderat
3 = puternic
"""
AXA_2_DEFINITION = """
fear măsoară dacă textul exprimă frică, panică, amenințare sau pericol iminent.
0 = absent
1 = slab
2 = moderat
3 = puternic
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.

Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [11]:
MINI_PROMPT = f"""
Ești un specialist în analiza discursului politic românesc prezent pe YouTube
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2 / 3
{AXA_2} = 0 / 1 / 2 / 3
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = slab, 2 = moderat, 3 = puternic
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un specialist în analiza discursului politic românesc prezent pe YouTube
SARCINĂ:
Adnotează comentariul folosind două axe:
1. people_vs_elite
2. fear
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
people_vs_elite = 0 / 1 / 2 / 3
fear = 0 / 1 / 2 / 3
DEFINIȚII:

people_vs_elite măsoară dacă textul construiește o opoziție între „popor/oameni”
și „elite/sistem/clasa conducătoare”.
0 = absent
1 = slab
2 = moderat
3 = puternic


fear măsoară dacă textul exprimă frică, panică, amenințare sau pericol iminent.
0 = absent
1 = slab
2 = moderat
3 = puternic

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru 

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [17]:
TESTS = corpus.sample(10)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
194,yt_Vy0H__Pk540_UgwcfULwNSmwJx70a794AaABAg,@CălinGeorgescu-CanalulOficial,Călin Georgescu - Acest blestem se va rupe ( 1...,Mulțumim pentru Istoria retrăită! Care se pare...
237,yt_whXg_RKoyoU_Ugxa6w6U31S5U0xhKZZ4AaABAg,RadioClasic,Lentila de contact cu Stelian Tănase - Radiogr...,MULTĂ LUME ȘI MAI ALES AMERICANI--CRED CĂ PEST...
209,yt_iH8jB4NlV9Y_UgzdHkRzIyxv2dqnBwF4AaABAg,georgesimionoficial,Episodul 2: Cum ne-au furat alegerile - Turism...,"Un video slab documentat, cu zero dovezi, doar..."
293,yt_0t6khZTkmxM_UgyAvGuYhrMIKsHLnXN4AaABAg,TuDecizi-s3g,Tu Decizi Live,"Aloooo,AUR ati gesit sistematic! De ce nu cere..."
258,yt_dESUmtdVSzo_UgxC_NsXxPQE5DWaQfN4AaABAg,turcescu111,"Războiul pustiește, dolarul crește, economia s...",Turcule las-o mai moale cu propaganda asta de ...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [21]:
USE_GEMINI = False
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: deepseek-chat


In [22]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [23]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
Mulțumim pentru Istoria retrăită! Care se pare ca s-a uitat plus Dumneavoastră sunteți Istorie CG11 ❤️🇷🇴🙏🏼

OUTPUT MODEL:
{
  "target": "Călin Georgescu",
  "stance": "pro",
  "tone": "afectiv",
  "people_vs_elite": 0,
  "fear": 0
}
COMENTARIU:
MULTĂ LUME ȘI MAI ALES AMERICANI--CRED CĂ PESTE 60% DINTRE AMERICANI--CONSIDERĂ CĂ TRUMP ARE ȘI PROBLEME DE SĂNĂTATE MENTALĂ, AMERICANII---UN POPOR CARE ARE DEMOCRAȚIA ÎN SÂNGE ÎL VOR ELIMINA ÎNTR-UN FEL ÎNAINTE DE A FACE MULT RĂU ,AMRICANILOR DAR ȘI ÎNTREGII OMENIRI!!!!

OUTPUT MODEL:
{
  "target": "Donald Trump",
  "stance": "anti",
  "tone": "acuzator",
  "people_vs_elite": 0,
  "fear": 2
}
COMENTARIU:
Un video slab documentat, cu zero dovezi, doar cu membrii AUR care spun cate si mai cate. Aici sunt 2 variante: fie faceti puscarie pentru astfel de continut mincinos, fie aveti dreptate, iar sistemul va baga la puscarie pentru ca ati demonstrat cat de puternic este.

OUTPUT MODEL:
{
  "target": "AUR",
  "stance": "anti",
  "tone": 

## Pasul 6 — Interpretare scurtă

### Axe: people_vs_elite si fear. Le-am ales pentru ca apar destul de frecvent in discursul politic online. Modelul a returnat JSON corect, nu am identificat probleme semnificative. 
### As simplifica promptul si as oferi mai multe exemple contrastante